In [ ]:
%env AWS_PROFILE=platform-developer

In [ ]:
import json
import pickle

import polars as pl

SIERRA_RAW_WORKS_PATH = "data/sierra-raw-works.parquet"
SIERRA_SOURCE_WORKS_PATH = "data/sierra-source-works.parquet"
FOLIO_RAW_BIBS_PATH = "data/folio-raw-bibs.parquet"
FOLIO_SOURCE_WORKS_PATH = "data/folio-source-works.parquet"

def from_parquet_snapshot(path: str):
    df = pl.read_parquet(path)
    return {row["id"]: pickle.loads(row["body"]) for row in df.iter_rows(named=True)}

In [ ]:
sierra_raw_works = {
    row["id"]: json.loads(row["body"])
    for row in pl.read_parquet(SIERRA_RAW_WORKS_PATH).iter_rows(named=True)
}
print(f"Sierra raw works count: {len(sierra_raw_works)}")

In [ ]:
folio_raw_bibs = dict(
    pl.read_parquet(FOLIO_RAW_BIBS_PATH, columns=["id", "content"]).iter_rows()
)
print(f"Folio raw bibs count: {len(folio_raw_bibs)}")

In [ ]:
sierra_source_works = from_parquet_snapshot(SIERRA_SOURCE_WORKS_PATH)
print(f"Sierra source works count: {len(sierra_source_works)}")

In [ ]:
folio_source_works = from_parquet_snapshot(FOLIO_SOURCE_WORKS_PATH)
print(f"Folio source works count: {len(folio_source_works)}")

In [ ]:
from adapters.extractors.oai_pmh.folio import config as folio_config
from adapters.extractors.oai_pmh.folio.runtime import FOLIO_CONFIG

oai_client = FOLIO_CONFIG.build_oai_client()

def get_raw_folio_record(instance_id: str):
    return oai_client.get_record(
        instance_id, metadata_prefix=folio_config.OAI_METADATA_PREFIX
    )

In [ ]:
folio_id_to_sierra_id = {}
for work_id, work in folio_source_works.items():
    if work.state.predecessor_identifier:
        sierra_id = work.state.predecessor_identifier.value
        folio_id_to_sierra_id[work_id] = sierra_id

# One Sierra bib resolves to one live Folio instance, so this reverse map is not
# lossy: the raw snapshot is a fresh OAI harvest, and instances superseded by an
# earlier Folio load are no longer in the repository. Build folio_source_works from
# the adapter store instead and it does become lossy - that table keeps every
# harvest generation, 2-3 rows per bib.
sierra_id_to_folio_id = {}
for k, v in folio_id_to_sierra_id.items():
    sierra_id_to_folio_id[v] = k

In [ ]:
def standardise_works_permanent(sierra_work, folio_work):
    """Normalise fields which we expect to differ between Folio and Sierra"""

    # Source identifiers will always be different
    del sierra_work["state"]["sourceIdentifier"]
    del folio_work["state"]["sourceIdentifier"]

    # Works no longer store siblings/children.
    # Popped rather than deleted: unlike CALM, most Sierra works are not hierarchical
    # and may carry no relations at all.
    relations = sierra_work["state"].get("relations", {})
    relations.pop("siblingsPreceding", None)
    relations.pop("siblingsSucceeding", None)
    relations.pop("children", None)

    # Source modified time will always be different
    del sierra_work["state"]["sourceModifiedTime"]
    del folio_work["state"]["sourceModifiedTime"]

    # Only Folio works have a predecessor identifier
    del folio_work["state"]["predecessorIdentifier"]

    # Only Folio works have a modified time
    del folio_work["state"]["modifiedTime"]

    # Versions will always be different
    del folio_work["version"]
    del sierra_work["version"]

    # Added automatically by the Elasticsearch ingest pipeline
    if "indexed_at" in sierra_work:
        del sierra_work["indexed_at"]

    # The Python pipeline uses the 'type' property more consistently than the Scala pipeline.
    for candidate in folio_work["state"].get("mergeCandidates", []):
        candidate["id"].pop("type", None)
    for subject in folio_work["data"].get("subjects", []):
        subject.pop("type", None)
    for item in folio_work["data"].get("production", []):
        for date in item.get("dates", []):
            date.pop("type", None)

    return sierra_work, folio_work


def standardise_works(sierra_work, folio_work):
    """Normalise some prevalent known issues/discrepancies to remove noise from the diff display."""

    sierra_work, folio_work = standardise_works_permanent(sierra_work, folio_work)
    # Add Sierra-specific normalisations here as the diff output reveals them, the way
    # the Axiell notebook accumulated the CALM whitespace/ampersand fixes.
    return sierra_work, folio_work


# Folio works whose Sierra bib is absent from the snapshot. Expected: the Sierra
# snapshot is a point-in-time dump and some 907 b-numbers have no Sierra record at all.
missing_sierra_works = []


def stream_normalised_source_works(limit: int | None = None):
    for i, (work_id, work) in enumerate(folio_source_works.items()):
        if limit and i > limit:
            break

        if i % 10000 == 0 and i > 0:
            print(f"Streamed {i} works")

        if work.type == "Deleted" or not work.state.predecessor_identifier:
            continue

        if work.state.predecessor_identifier:
            sierra_id = work.state.predecessor_identifier.value
            sierra_work = sierra_source_works.get(sierra_id)

            if sierra_work is None:
                missing_sierra_works.append(work_id)
                continue

            if sierra_work["type"] in {"Deleted", "Invisible"}:
                continue

            sierra_work, folio_work = standardise_works(copy.deepcopy(sierra_work), work.model_dump(exclude_none=True))
            yield {"id": sierra_id, "work": sierra_work}, {"id": work_id, "work": folio_work}

In [ ]:
from deepdiff import DeepDiff
import re
import copy

matching_works = 0
diff_works = 0

for sierra, folio in stream_normalised_source_works(10000):
    sierra_id, sierra_work = sierra["id"], sierra["work"]
    folio_id, folio_work = folio["id"], folio["work"]

    diffs = DeepDiff(sierra_work, folio_work, ignore_order=True)
    if diffs:
        print(folio_id, sierra_id)
        diff_works += 1

        if "values_changed" in diffs:
            for key, val in diffs["values_changed"].items():
                print(key)
                print(f'\t {val["old_value"]}')
                print(f'\t {val["new_value"]}')
        if "iterable_item_removed" in diffs:
            print("ITEM REMOVED")
            print(diffs["iterable_item_removed"])
        if "iterable_item_added" in diffs:
            print("ITEM ADDED")
            print(diffs["iterable_item_added"])
        if "dictionary_item_removed" in diffs:
            print("DICTIONARY ITEM REMOVED")
            print(diffs["dictionary_item_removed"])
        if "dictionary_item_added" in diffs:
            print("DICTIONARY ITEM ADDED")
            print(diffs["dictionary_item_added"])

        print("------------------------------------")
    else:
        matching_works += 1

print(f"Matching works: {matching_works}")
print(f"Diff works: {diff_works}")
print(f"Folio works whose Sierra bib is missing from the snapshot: {len(missing_sierra_works)}")

In [ ]:
deleted_sierra_works = []
deleted_sierra_works_without_folio_match = []
sierra_works_without_folio_match = []
deleted_folio_works = []
no_predecessor_id = []
sierra_id_not_in_snapshot = []

for work_id, work in folio_source_works.items():
    if work.type == "Deleted":
        deleted_folio_works.append(work_id)
        continue

    if work.state.predecessor_identifier:
        sierra_id = work.state.predecessor_identifier.value
        sierra_work = sierra_source_works.get(sierra_id)

        if sierra_work is None:
            sierra_id_not_in_snapshot.append(work_id)
            continue

        if sierra_work["type"] == "Deleted":
            deleted_sierra_works.append(work_id)
            continue
    else:
        no_predecessor_id.append(work_id)

for work_id, work in sierra_source_works.items():
    if work_id in sierra_id_to_folio_id:
        continue

    if work["type"] == "Deleted":
        deleted_sierra_works_without_folio_match.append(work_id)
    else:
        sierra_works_without_folio_match.append(work_id)

print(f"Works deleted in Folio: {len(deleted_folio_works)}")
print(f"Works deleted in Sierra but visible in Folio: {len(deleted_sierra_works)}")
print(f"Folio works without a predecessor identifier: {len(no_predecessor_id)}")
print(f"Folio works whose Sierra bib is missing from the snapshot: {len(sierra_id_not_in_snapshot)}")

print(f"Deleted Sierra works without a matching Folio work: {len(deleted_sierra_works_without_folio_match)}")
print(f"Non-deleted Sierra works without a matching Folio work: {len(sierra_works_without_folio_match)}")